# Titanic Survival Prediction — Exploratory Data Analysis

**Mini Project | Computer Engineering**

This notebook walks through the complete data analysis pipeline:
1. Dataset loading and initial inspection
2. Missing-value analysis
3. Exploratory data analysis (EDA)
4. Feature engineering
5. Model training and comparison
6. Evaluation and conclusions

---
> **Prerequisite:** Place `train.csv` and `test.csv` from the [Kaggle Titanic competition](https://www.kaggle.com/competitions/titanic/data) in the `../data/` directory before running this notebook.

## 1. Imports and Configuration

In [ ]:
import sys
import pathlib
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so we can import src/
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    load_dataset, drop_duplicates,
    get_missing_summary
)
from src.features import engineer_features, ALL_FEATURES, TARGET
from src.model_training import (
    train_all_models, select_best_model,
    save_pipeline, build_metrics_dataframe
)
from src.evaluation import (
    evaluate_model, plot_confusion_matrix,
    plot_model_comparison, plot_feature_importance
)

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)

DATA_DIR = pathlib.Path('../data')
print('Setup complete ✓')

## 2. Load Dataset

In [ ]:
train_raw = load_dataset(DATA_DIR / 'train.csv')
test_raw  = load_dataset(DATA_DIR / 'test.csv')

print(f'Training set : {train_raw.shape[0]} rows × {train_raw.shape[1]} columns')
print(f'Test set     : {test_raw.shape[0]} rows × {test_raw.shape[1]} columns')

## 3. Initial Inspection

In [ ]:
train_raw.head(10)

In [ ]:
train_raw.info()

In [ ]:
train_raw.describe(include='all')

## 4. Missing-Value Analysis

Understanding what is missing is crucial before choosing an imputation strategy.

In [ ]:
missing_summary = get_missing_summary(train_raw)
print('Missing values in training set:')
display(missing_summary)

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(missing_summary.index, missing_summary['Missing %'], color=sns.color_palette('Set2'))
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Data by Column (Training Set)')
plt.tight_layout()
plt.show()

**Observations:**
- **Age** is missing in ~20% of rows → impute with median (robust to outliers)
- **Cabin** is missing in ~77% of rows → too sparse to use directly; drop after extracting deck if needed
- **Embarked** is missing in only 2 rows → impute with mode (most frequent port)

All statistics (median, mode) are computed on the **full training set** and then applied to validation/test. This prevents data leakage.

## 5. Exploratory Data Analysis

In [ ]:
# 5a. Overall survival distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

survival_counts = train_raw['Survived'].value_counts()
axes[0].pie(
    survival_counts.values,
    labels=['Did Not Survive', 'Survived'],
    colors=['#e74c3c', '#2ecc71'],
    autopct='%1.1f%%', startangle=90
)
axes[0].set_title('Overall Survival Rate')

sns.countplot(data=train_raw, x='Survived', hue='Pclass', ax=axes[1],
              palette='Set2')
axes[1].set_xticklabels(['Did Not Survive', 'Survived'])
axes[1].set_title('Survival Count by Passenger Class')
axes[1].legend(title='Class')

plt.tight_layout()
plt.show()

print(f"Survival rate: {train_raw['Survived'].mean()*100:.1f}%")

In [ ]:
# 5b. Survival by gender
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gender_survival = train_raw.groupby('Sex')['Survived'].mean() * 100
gender_survival.plot(kind='bar', ax=axes[0], color=['#3498db', '#e91e8c'],
                     edgecolor='white', rot=0)
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('Survival Rate by Sex')
for bar, val in zip(axes[0].patches, gender_survival.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center')

sns.countplot(data=train_raw, x='Sex', hue='Survived', ax=axes[1], palette='Set2')
axes[1].set_title('Count by Sex and Survival')
axes[1].legend(['Did Not Survive', 'Survived'])

plt.tight_layout()
plt.show()

female_rate = train_raw[train_raw['Sex']=='female']['Survived'].mean()*100
male_rate   = train_raw[train_raw['Sex']=='male']['Survived'].mean()*100
print(f'Female survival rate : {female_rate:.1f}%')
print(f'Male survival rate   : {male_rate:.1f}%')

In [ ]:
# 5c. Age distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_raw[train_raw['Survived']==1]['Age'].hist(
    bins=30, ax=axes[0], alpha=0.7, color='#2ecc71', label='Survived')
train_raw[train_raw['Survived']==0]['Age'].hist(
    bins=30, ax=axes[0], alpha=0.7, color='#e74c3c', label='Did Not Survive')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution by Survival')
axes[0].legend()

# Violin plot
sns.violinplot(data=train_raw.dropna(subset=['Age']),
               x='Survived', y='Age', hue='Sex',
               ax=axes[1], palette=['#3498db', '#e91e8c'], split=True)
axes[1].set_xticklabels(['Did Not Survive', 'Survived'])
axes[1].set_title('Age Distribution by Survival and Sex')

plt.tight_layout()
plt.show()

In [ ]:
# 5d. Fare and embarkation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=train_raw, x='Pclass', y='Fare',
            hue='Survived', ax=axes[0], palette='Set2')
axes[0].set_ylim(0, 300)
axes[0].set_title('Fare by Class and Survival (clipped at £300)')
axes[0].legend(['Did Not Survive', 'Survived'])

emb_surv = train_raw.groupby('Embarked')['Survived'].mean() * 100
emb_surv.index = ['Cherbourg (C)' if p=='C' else 'Queenstown (Q)'
                  if p=='Q' else 'Southampton (S)' for p in emb_surv.index]
emb_surv.plot(kind='bar', ax=axes[1],
              color=sns.color_palette('Set2'), edgecolor='white', rot=0)
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_title('Survival Rate by Embarkation Port')
axes[1].set_ylim(0, 70)

plt.tight_layout()
plt.show()

In [ ]:
# 5e. Correlation heatmap (numeric columns)
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr = train_raw[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Pearson Correlation Heatmap')
plt.tight_layout()
plt.show()

**Observations from correlation heatmap:**
- `Pclass` has the strongest negative correlation with `Survived` (-0.34) — lower class = lower survival
- `Fare` has a positive correlation (+0.26) — higher fare linked with higher class and better survival
- `Age` has a slight negative correlation — older passengers slightly less likely to survive
- `SibSp` and `Parch` have small correlations — family size needs feature engineering to be informative

## 6. Feature Engineering

In [ ]:
# Keep missing values for modelling: the scikit-learn ColumnTransformer
# fits median/most-frequent imputers using each model's training fold only.
train_clean = train_raw.copy()
print('Missing values are handled inside each fitted model pipeline.')

In [ ]:
# Engineer features
train_fe = engineer_features(train_clean)
# Re-attach target (engineer_features does not drop it if present)
if 'Survived' not in train_fe.columns:
    train_fe['Survived'] = train_clean['Survived'].values

print('Feature-engineered columns:')
print(list(train_fe.columns))
train_fe.head()

In [ ]:
# Visualise new features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Title vs Survival
title_surv = train_fe.groupby('Title')['Survived'].mean() * 100
title_surv.sort_values().plot(kind='barh', ax=axes[0],
                               color=sns.color_palette('Set2'))
axes[0].set_xlabel('Survival Rate (%)')
axes[0].set_title('Survival Rate by Title')

# FamilySize vs Survival
fam_surv = train_fe.groupby('FamilySize')['Survived'].mean() * 100
fam_surv.plot(kind='bar', ax=axes[1],
               color=sns.color_palette('Blues_r', len(fam_surv)), rot=0)
axes[1].set_xlabel('Family Size')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_title('Survival Rate by Family Size')

# IsAlone vs Survival
alone_surv = train_fe.groupby('IsAlone')['Survived'].mean() * 100
alone_surv.index = ['With Family', 'Alone']
alone_surv.plot(kind='bar', ax=axes[2],
                color=['#2ecc71', '#e74c3c'], edgecolor='white', rot=0)
axes[2].set_ylabel('Survival Rate (%)')
axes[2].set_title('Survival Rate: Alone vs With Family')

plt.tight_layout()
plt.show()

**Feature engineering insights:**
- Compare the calculated title survival rates in the chart; the feature encodes information from passenger names.
- Compare the calculated FamilySize rates to see whether small or large groups differ in this file.
- Compare the calculated IsAlone rates to inspect the relationship between travelling alone and survival.

> These are descriptive patterns calculated from the downloaded file, not universal rules.

## 7. Model Training

In [ ]:
print('Training all models ...')
pipelines, metrics, X_train, y_train, X_val, y_val = train_all_models(train_fe)
print('Done ✓')

## 8. Model Comparison

In [ ]:
metrics_df = build_metrics_dataframe(metrics)
display(metrics_df)

In [ ]:
# Comparison bar chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
melted = metrics_df.melt(id_vars='Model', value_vars=metrics_to_plot,
                          var_name='Metric', value_name='Score')

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=melted, x='Model', y='Score', hue='Metric',
            palette='Set2', ax=ax, edgecolor='white')
ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison — Classification Metrics', fontsize=14)
ax.legend(title='Metric', loc='lower right')
plt.tight_layout()
plt.show()

## 9. Evaluation and Conclusions

In [ ]:
best_name = select_best_model(metrics)
best_pipeline = pipelines[best_name]
print(f'Best model by F1 score: {best_name}')
print(f"F1 = {metrics[best_name]['f1']:.4f}")
print()
print('Classification Report:')
print(metrics[best_name]['report'])

In [ ]:
# Confusion matrix for best model
cm = metrics[best_name]['confusion_matrix']
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'],
            ax=ax, linewidths=0.5)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (tree-based models)
clf = best_pipeline.named_steps['classifier']
if hasattr(clf, 'feature_importances_'):
    preprocessor = best_pipeline.named_steps['preprocessor']
    enc_features = preprocessor.get_feature_names_out()
    importances = clf.feature_importances_
    indices = importances.argsort()[::-1][:15]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(len(indices)), importances[indices][::-1],
            color=sns.color_palette('Blues_d', len(indices)))
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([enc_features[i] for i in indices[::-1]])
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 15 Feature Importances — {best_name}')
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_name} does not expose feature importances.')

In [ ]:
# Save the best pipeline
saved = save_pipeline(best_pipeline, best_name)
print(f'Pipeline saved → {saved}')

## Conclusions

- The charts above calculate survival patterns by gender, class, age, fare, port, title, and family structure from the supplied file.
- The model comparison table identifies the preferred classifier using the measured validation F1-score for this run.
- Feature engineering makes family structure and title available to the classifiers without using passenger identifiers as model inputs.

### Limitations
- Missing numeric and categorical values are imputed inside each model pipeline. The imputers are fitted only on the training fold, then applied to validation data.
- The Cabin column's missingness and its treatment are shown by the inspection cells; it is excluded from the model feature list.
- The model predicts aggregate probability patterns, not individual fates.
- The training set size is displayed by the loading cell; this is a small dataset for machine learning.